# Assignment 2: Autonomous Mumbai Trip Planning Agent

**Topic:** Comparing CoT, ToT, and ReAct strategies  
**Task:** Plan a 3-day winter trip from Delhi to Mumbai for 3 people within â‚¹40,000.

## Goal
Build and compare three LLM-based planning strategies:
1. Chain-of-Thought (CoT)
2. Tree-of-Thought (ToT)
3. ReAct (Reason + Act)

## Constraints
- Starting city: Delhi
- Destination: Mumbai
- Duration: 3 days / 2 nights
- Travellers: 3
- Total budget: â‚¹40,000
- Season: Winter
- Budget includes travel, accommodation, food, local transport, sightseeing, and buffer

## Mid-task disruption
The budget is reduced from â‚¹40,000 to â‚¹32,000.  
Each strategy must revise its plan and we will compare its ability to adapt.

In [1]:
# Core Python imports
import os
import json
import time
from datetime import datetime

# Trip-planning task specification
TRIP_CONFIG = {
    "origin": "Delhi",
    "destination": "Mumbai",
    "travel_season": "Winter",
    "travel_dates": "To be selected",
    "travellers": 3,
    "days": 3,
    "nights": 2,
    "total_budget_inr": 40000,
    "budget_includes": [
        "return intercity travel",
        "accommodation",
        "meals",
        "local transport",
        "sightseeing",
        "emergency buffer"
    ],
    "preferences": [
        "budget-friendly",
        "realistic travel times",
        "winter-suitable activities",
        "safe and practical itinerary"
    ],
    "disruption": {
        "type": "budget_reduction",
        "new_budget_inr": 32000,
        "description": "The trip budget is reduced after the first itinerary is created."
    }
}

print(json.dumps(TRIP_CONFIG, indent=2))

{
  "origin": "Delhi",
  "destination": "Mumbai",
  "travel_season": "Winter",
  "travel_dates": "To be selected",
  "travellers": 3,
  "days": 3,
  "nights": 2,
  "total_budget_inr": 40000,
  "budget_includes": [
    "return intercity travel",
    "accommodation",
    "meals",
    "local transport",
    "sightseeing",
    "emergency buffer"
  ],
  "preferences": [
    "budget-friendly",
    "realistic travel times",
    "winter-suitable activities",
    "safe and practical itinerary"
  ],
  "disruption": {
    "type": "budget_reduction",
    "new_budget_inr": 32000,
    "description": "The trip budget is reduced after the first itinerary is created."
  }
}


In [2]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

COHERE_API_KEY = os.getenv("COHERE_API_KEY")

print("Cohere API key loaded:", COHERE_API_KEY is not None)

Cohere API key loaded: True


In [3]:
import cohere

co = cohere.ClientV2(api_key=COHERE_API_KEY)

In [4]:
response = co.chat(
    model="command-a-plus-05-2026",
    messages=[
        {"role": "user", "content": "Hello! Give me a one-line introduction."}
    ]
)

print(response.message.content)

[ThinkingAssistantMessageResponseContentItem(type='thinking', thinking='The user asks: "Hello! Give me a one-line introduction." So we need to respond with a one-line introduction. Probably something like "I\'m Command, a Cohere-powered language model, here to help." That\'s one line. Should be concise. Provide one line.'), TextAssistantMessageResponseContentItem(type='text', text="I'm Command, a Cohere-powered language model, ready to assist you with concise and insightful answers.")]


## Shared Agent Utilities

All strategies use the same trip constraints, model, and output requirements.
Only the reasoning strategy changes.

In [5]:
def build_trip_prompt(trip_config):
    """Creates the identical task description used by every strategy."""
    
    return f"""
You are an expert budget travel-planning agent.

Create a realistic winter trip plan using these constraints:

- Origin: {trip_config["origin"]}
- Destination: {trip_config["destination"]}
- Travellers: {trip_config["travellers"]}
- Duration: {trip_config["days"]} days / {trip_config["nights"]} nights
- Maximum total budget: INR {trip_config["total_budget_inr"]}
- Budget includes: {", ".join(trip_config["budget_includes"])}
- Preferences: {", ".join(trip_config["preferences"])}

Your final answer must contain:
1. A day-wise itinerary
2. An itemized budget breakdown
3. A total estimated cost in INR
4. A short explanation showing how all constraints are satisfied

Important:
- Keep the total within the stated budget.
- Use realistic travel times and practical activities.
- Clearly mention assumptions when exact prices are unavailable.
"""

BASE_TRIP_PROMPT = build_trip_prompt(TRIP_CONFIG)

print(BASE_TRIP_PROMPT)


You are an expert budget travel-planning agent.

Create a realistic winter trip plan using these constraints:

- Origin: Delhi
- Destination: Mumbai
- Travellers: 3
- Duration: 3 days / 2 nights
- Maximum total budget: INR 40000
- Budget includes: return intercity travel, accommodation, meals, local transport, sightseeing, emergency buffer
- Preferences: budget-friendly, realistic travel times, winter-suitable activities, safe and practical itinerary

Your final answer must contain:
1. A day-wise itinerary
2. An itemized budget breakdown
3. A total estimated cost in INR
4. A short explanation showing how all constraints are satisfied

Important:
- Keep the total within the stated budget.
- Use realistic travel times and practical activities.
- Clearly mention assumptions when exact prices are unavailable.



In [6]:
def extract_response_text(response):
    """Extracts only the final visible answer, excluding thinking content."""
    
    text_parts = []
    
    for item in response.message.content:
        if hasattr(item, "text"):
            text_parts.append(item.text)
    
    return "\n".join(text_parts)


def run_base_agent(system_prompt, user_prompt):
    """
    Sends a request to Command A+ and records observable performance data.
    This function will be reused by CoT, ToT, and ReAct.
    """
    
    start_time = time.time()
    
    response = co.chat(
        model="command-a-plus-05-2026",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    
    elapsed_seconds = round(time.time() - start_time, 2)
    final_answer = extract_response_text(response)
    
    return {
        "final_answer": final_answer,
        "execution_time_seconds": elapsed_seconds,
        "raw_response": response
    }

In [7]:
SYSTEM_PROMPT = """
You are a helpful AI travel-planning assistant.
Follow the user's instructions carefully and provide a clear, practical answer.
"""

test_result = run_base_agent(
    system_prompt=SYSTEM_PROMPT,
    user_prompt="In one sentence, confirm that you can help plan the Mumbai trip."
)

print(test_result["final_answer"])
print(f"\nExecution time: {test_result['execution_time_seconds']} seconds")

Yes, I can definitely help you plan your Mumbai trip.

Execution time: 0.51 seconds


## Strategy 1: Chain-of-Thought (CoT) Planning Agent

The CoT agent follows one structured planning path:
trip requirements â†’ budget allocation â†’ itinerary â†’ constraint check.

It does not explore alternatives or call tools.

In [8]:
COT_SYSTEM_PROMPT = """
You are a budget travel-planning agent using a linear Chain-of-Thought strategy.

Work through the task in one structured path:
1. Identify trip requirements
2. Allocate the available budget
3. Create a practical day-wise itinerary
4. Check the final budget and constraints

Do not explore multiple competing plans.
Do not use external tools or claim to have searched the web.
If a price is uncertain, make a reasonable estimate and label it as an assumption.
"""


def run_cot_agent(trip_config):
    cot_user_prompt = f"""
{build_trip_prompt(trip_config)}

Use a single linear planning approach.

Return your answer using exactly these headings:

## Planning Summary
Briefly explain the main planning stages you followed.

## Budget Allocation
Show estimated costs for return travel, accommodation, meals,
local transport, sightseeing, and buffer.

## Day-wise Itinerary
Give a practical plan for Day 1, Day 2, and Day 3.

## Constraint Check
Confirm the traveller count, duration, winter suitability,
and whether the total stays within INR {trip_config["total_budget_inr"]}.

## Assumptions
List any estimated prices or assumptions.
"""

    result = run_base_agent(
        system_prompt=COT_SYSTEM_PROMPT,
        user_prompt=cot_user_prompt
    )

    result["strategy"] = "CoT"
    result["planning_steps"] = 4

    return result

In [9]:
cot_initial_result = run_cot_agent(TRIP_CONFIG)

print(cot_initial_result["final_answer"])
print(f"\nStrategy: {cot_initial_result['strategy']}")
print(f"Planning steps: {cot_initial_result['planning_steps']}")
print(f"Execution time: {cot_initial_result['execution_time_seconds']} seconds")

## Planning Summary
I first listed the trip requirements (origin, destination, travellers, dates, budget, inclusions).  
Next I allocated the INR 40 000 budget to the major cost blocks: return travel, lodging, meals, local transport, sightseeing and a safety buffer.  
Using realistic prices for trains, budget hotels, meals and Mumbai attractions, I built a compact 3‑day/2‑night itinerary that fits the travel times and offers winter‑friendly activities.  
Finally I verified that all constraints (traveller count, duration, winter suitability, total cost) are met and noted any assumptions.

## Budget Allocation
| Category | Estimated Cost (INR) | Notes |
|----------|----------------------|-------|
| **Return Intercity Travel** (Delhi ↔ Mumbai, AC 3‑tier, 3 pax) | 15 000 | INR 2 500 per person per leg × 2 legs × 3 pax |
| **Accommodation** (2 nights, triple occupancy budget hotel/ guesthouse) | 1 600 | INR 800 per night |
| **Meals** (3 pax, 3 days, 3 meals/day) | 3 240 | INR 120 per meal 

## Strategy 2: Tree-of-Thought (ToT) Planning Agent

The ToT agent creates multiple candidate itineraries, evaluates them against
the same constraints, and selects the strongest plan.

In [10]:
TOT_SYSTEM_PROMPT = """
You are a budget travel-planning agent using a Tree-of-Thought strategy.

Generate three distinct candidate trip plans before selecting one:
1. Budget-first plan
2. Balanced plan
3. Comfort-first plan

Evaluate each plan for budget compliance, practicality, travel time,
and winter suitability. Then choose the best plan.

Do not use external tools or claim to have searched the web.
Clearly label uncertain prices as estimates.
"""


def run_tot_agent(trip_config):
    tot_user_prompt = f"""
{build_trip_prompt(trip_config)}

Create exactly three candidate Mumbai-trip plans:

- Branch A: lowest-cost / budget-first
- Branch B: balanced cost and experience
- Branch C: more comfort, while trying to stay within budget

Return your response using exactly these headings:

## Candidate Plans
Describe Branch A, Branch B, and Branch C with their estimated total costs.

## Branch Evaluation
Compare all three branches using:
- budget compliance
- practical travel time
- winter suitability
- overall trip experience

## Selected Plan
Choose the best branch and explain why.

## Final Day-wise Itinerary
Give the selected Day 1, Day 2, and Day 3 plan.

## Final Budget Breakdown
Show all major costs and the total.

## Assumptions
List estimated or uncertain prices.
"""

    result = run_base_agent(
        system_prompt=TOT_SYSTEM_PROMPT,
        user_prompt=tot_user_prompt
    )

    result["strategy"] = "ToT"
    result["candidate_branches"] = 3
    result["planning_steps"] = 6

    return result

In [11]:
tot_initial_result = run_tot_agent(TRIP_CONFIG)

print(tot_initial_result["final_answer"])
print(f"\nStrategy: {tot_initial_result['strategy']}")
print(f"Candidate branches: {tot_initial_result['candidate_branches']}")
print(f"Execution time: {tot_initial_result['execution_time_seconds']} seconds")

## Candidate Plans
**Branch A – Lowest‑Cost / Budget‑First**  
- Intercity travel: Bus (AC sleeper) Delhi ↔ Mumbai, ₹800 × 2 × 3 = **₹4,800**  
- Accommodation: Budget guesthouse, triple room, ₹1,500 /night × 2 nights = **₹3,000**  
- Meals: ₹500 /person /day × 3 days × 3 travellers = **₹4,500**  
- Local transport: ₹300 /day × 3 days × 3 travellers = **₹900**  
- Sightseeing: ₹300 /person /day × 3 days × 3 travellers = **₹2,700**  
- Emergency buffer (5 % of subtotal): **₹795**  
- **Estimated total:** **≈ ₹16,700**  

**Branch B – Balanced Cost & Experience**  
- Intercity travel: Train (3‑tier AC) Delhi ↔ Mumbai, ₹1,500 × 2 × 3 = **₹9,000**  
- Accommodation: Mid‑range hotel, triple room, ₹2,500 /night × 2 nights = **₹5,000**  
- Meals: ₹800 /person /day × 3 days × 3 travellers = **₹7,200**  
- Local transport: ₹500 /day × 3 days × 3 travellers = **₹1,500**  
- Sightseeing: ₹500 /person /day × 3 days × 3 travellers = **₹4,500**  
- Emergency buffer (5 % of subtotal): **₹1,360**  
- 

In [12]:
import json
from tavily import TavilyClient


def web_search(query, max_results=5):
    """Searches the web and returns compact, useful result data."""
    
    tavily_api_key = os.getenv("TAVILY_API_KEY")

    if not tavily_api_key:
        raise ValueError("TAVILY_API_KEY is missing from the .env file.")

    tavily = TavilyClient(api_key=tavily_api_key)

    search_response = tavily.search(
        query=query,
        max_results=max_results,
        search_depth="basic"
    )

    results = []

    for item in search_response.get("results", []):
        results.append({
            "title": item.get("title", ""),
            "url": item.get("url", ""),
            "content": item.get("content", "")
        })

    return results


def calculate_trip_budget(
    return_travel_per_person,
    accommodation_total,
    meals_total,
    local_transport_total,
    sightseeing_total,
    buffer_total,
    travellers=3
):
    """Calculates a transparent trip budget."""
    
    total = (
        return_travel_per_person * travellers
        + accommodation_total
        + meals_total
        + local_transport_total
        + sightseeing_total
        + buffer_total
    )

    return [{
        "return_travel_per_person_inr": return_travel_per_person,
        "travellers": travellers,
        "return_travel_total_inr": return_travel_per_person * travellers,
        "accommodation_total_inr": accommodation_total,
        "meals_total_inr": meals_total,
        "local_transport_total_inr": local_transport_total,
        "sightseeing_total_inr": sightseeing_total,
        "buffer_total_inr": buffer_total,
        "grand_total_inr": total,
        "within_40000_budget": total <= 40000
    }]

In [13]:
REACT_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Searches the live web for current travel, hotel, transport, attraction, or weather information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "A focused travel-related web-search query."
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Maximum results to return, between 1 and 5."
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_trip_budget",
            "description": "Calculates the complete Mumbai trip budget in INR for all travellers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "return_travel_per_person": {"type": "number"},
                    "accommodation_total": {"type": "number"},
                    "meals_total": {"type": "number"},
                    "local_transport_total": {"type": "number"},
                    "sightseeing_total": {"type": "number"},
                    "buffer_total": {"type": "number"},
                    "travellers": {"type": "integer"}
                },
                "required": [
                    "return_travel_per_person",
                    "accommodation_total",
                    "meals_total",
                    "local_transport_total",
                    "sightseeing_total",
                    "buffer_total"
                ]
            }
        }
    }
]

FUNCTION_MAP = {
    "web_search": web_search,
    "calculate_trip_budget": calculate_trip_budget
}

In [14]:
REACT_SYSTEM_PROMPT = """
You are a ReAct travel-planning agent.

You must use tools when needed:
- Use web_search to collect current travel information.
- Use calculate_trip_budget before giving the final itinerary.

Work iteratively:
Reason about missing information, call an appropriate tool, inspect its
result, and revise the plan.

Do not invent web-search results or calculation results.
Use the final tool evidence in your answer.
"""


def run_react_agent(trip_config, max_iterations=4):
    react_user_prompt = f"""
{build_trip_prompt(trip_config)}

Use live web search to research relevant Mumbai travel information.
Then use the budget calculator tool before presenting the final plan.

Because exact travel dates are not yet fixed, treat prices as estimates,
not bookable quotations.

Return these final headings after using the tools:

## Tool Evidence
## Budget Breakdown
## Day-wise Itinerary
## Constraint Check
## Sources and Assumptions
"""

    messages = [
        {"role": "system", "content": REACT_SYSTEM_PROMPT},
        {"role": "user", "content": react_user_prompt}
    ]

    tool_log = []
    start_time = time.time()
    final_answer = ""

    for _ in range(max_iterations):
        response = co.chat(
            model="command-a-plus-05-2026",
            messages=messages,
            tools=REACT_TOOLS,
            temperature=0
        )

        tool_calls = response.message.tool_calls

        if not tool_calls:
            final_answer = extract_response_text(response)
            break

        messages.append(response.message)

        for tool_call in tool_calls:
            tool_name = tool_call.function.name
            tool_arguments = json.loads(tool_call.function.arguments)

            tool_result = FUNCTION_MAP[tool_name](**tool_arguments)

            tool_log.append({
                "tool_name": tool_name,
                "arguments": tool_arguments,
                "result": tool_result
            })

            tool_content = [
                {
                    "type": "document",
                    "document": {"data": json.dumps(item)}
                }
                for item in tool_result
            ]

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_content
            })

    elapsed_seconds = round(time.time() - start_time, 2)

        # If the agent used all tool iterations, force one final answer
    # using the collected tool results.
    if not final_answer:
        messages.append({
            "role": "user",
            "content": """
You have enough tool evidence now.
Do not call any more tools.

Create the final Mumbai trip plan using the required headings:
## Tool Evidence
## Budget Breakdown
## Day-wise Itinerary
## Constraint Check
## Sources and Assumptions
"""
        })

        final_response = co.chat(
            model="command-a-plus-05-2026",
            messages=messages,
            temperature=0
        )

        final_answer = extract_response_text(final_response)

    return {
        "strategy": "ReAct",
        "final_answer": final_answer,
        "tool_log": tool_log,
        "tool_calls_count": len(tool_log),
        "execution_time_seconds": elapsed_seconds
    }

In [15]:
load_dotenv(override=True)

if os.getenv("TAVILY_API_KEY"):
    print("Tavily API key loaded successfully.")
else:
    
    print("Tavily API key is still missing.")

Tavily API key loaded successfully.


In [16]:
react_initial_result = run_react_agent(TRIP_CONFIG)

print(react_initial_result["final_answer"])
print(f"\nTool calls made: {react_initial_result['tool_calls_count']}")
print(f"Execution time: {react_initial_result['execution_time_seconds']} seconds")

## Tool Evidence
I conducted web searches for Delhi-Mumbai travel costs, Mumbai accommodation prices, local transport fares, and sightseeing options. Key findings:
- Delhi-Mumbai train sleeper class: ~INR 1,500-2,000 per person round trip
- Mumbai budget accommodation: INR 800-1,200 per night for shared rooms
- Food costs: INR 400-600 daily per person for budget meals
- Elephanta Caves tour: INR 3,000-4,000 for 3 people
- Most Mumbai attractions are free or low-cost (INR 20-50 entry)

The budget calculator confirmed total estimated cost of INR 17,000, well within the INR 40,000 budget constraint.

## Budget Breakdown
| Category | Cost (INR) | Details |
|----------|------------|---------|
| Return Travel | 4,500 | Train sleeper class for 3 travelers (INR 1,500 each) |
| Accommodation | 2,000 | Budget guesthouse, 2 nights (INR 1,000/night for 3 people) |
| Meals | 3,000 | Breakfast, lunch, dinner + snacks for 3 travelers, 2 days |
| Local Transport | 1,000 | Metro, buses, auto-rickshaws 

## Dynamic Replanning Test

After the original â‚¹40,000 plan is generated, the total budget is reduced
to â‚¹32,000. Each strategy must create a revised feasible itinerary.

In [17]:
def build_trip_prompt(trip_config):
    """Creates the identical task description used by every strategy."""

    replanning_context = trip_config.get("replanning_context", "")

    return f"""
You are an expert budget travel-planning agent.

Create a realistic winter trip plan using these constraints:

- Origin: {trip_config["origin"]}
- Destination: {trip_config["destination"]}
- Travellers: {trip_config["travellers"]}
- Duration: {trip_config["days"]} days / {trip_config["nights"]} nights
- Maximum total budget: INR {trip_config["total_budget_inr"]}
- Budget includes: {", ".join(trip_config["budget_includes"])}
- Preferences: {", ".join(trip_config["preferences"])}

{replanning_context}

Your final answer must contain:
1. A day-wise itinerary
2. An itemized budget breakdown
3. A total estimated cost in INR
4. A short explanation showing how all constraints are satisfied

Important:
- Keep the total within the stated budget.
- Use realistic travel times and practical activities.
- Clearly mention assumptions when exact prices are unavailable.
"""

In [18]:
import copy


def create_disrupted_config(trip_config, new_budget):
    revised_config = copy.deepcopy(trip_config)

    original_budget = revised_config["total_budget_inr"]
    revised_config["total_budget_inr"] = new_budget

    revised_config["replanning_context"] = f"""
This is a replanning request.

An earlier plan was created with a budget of INR {original_budget}.
The available budget has now been reduced to INR {new_budget}.

Revise the plan to remain feasible. Clearly explain what was reduced,
replaced, or removed to meet the new budget.
"""

    return revised_config


DISRUPTED_TRIP_CONFIG = create_disrupted_config(
    TRIP_CONFIG,
    new_budget=32000
)

print(json.dumps(DISRUPTED_TRIP_CONFIG, indent=2))

{
  "origin": "Delhi",
  "destination": "Mumbai",
  "travel_season": "Winter",
  "travel_dates": "To be selected",
  "travellers": 3,
  "days": 3,
  "nights": 2,
  "total_budget_inr": 32000,
  "budget_includes": [
    "return intercity travel",
    "accommodation",
    "meals",
    "local transport",
    "sightseeing",
    "emergency buffer"
  ],
  "preferences": [
    "budget-friendly",
    "realistic travel times",
    "winter-suitable activities",
    "safe and practical itinerary"
  ],
  "disruption": {
    "type": "budget_reduction",
    "new_budget_inr": 32000,
    "description": "The trip budget is reduced after the first itinerary is created."
  },
  "replanning_context": "\nThis is a replanning request.\n\nAn earlier plan was created with a budget of INR 40000.\nThe available budget has now been reduced to INR 32000.\n\nRevise the plan to remain feasible. Clearly explain what was reduced,\nreplaced, or removed to meet the new budget.\n"
}


In [19]:
cot_replanned_result = run_cot_agent(DISRUPTED_TRIP_CONFIG)
tot_replanned_result = run_tot_agent(DISRUPTED_TRIP_CONFIG)
react_replanned_result = run_react_agent(DISRUPTED_TRIP_CONFIG)

In [20]:
print("=== CoT Replanned Result ===")
print(cot_replanned_result["final_answer"])

=== CoT Replanned Result ===
## Planning Summary
- Identified requirements: 3 travellers, 3 days/2 nights, Delhi → Mumbai winter trip, budget INR 32,000 covering travel, stay, meals, local transport, sightseeing, and buffer.  
- Allocated budget proportionally: travel 20 %, accommodation 20 %, meals 20 %, local transport 10 %, sightseeing 15 %, buffer 15 %.  
- Designed a day‑wise itinerary using sleeper‑class train, budget hotel, simple meals, Mumbai local passes and two paid attractions.  
- Verified all constraints and total cost stays well under INR 32,000.

## Budget Allocation
| Item | Details | Estimated Cost (INR) | Notes |
|------|---------|----------------------|-------|
| Return intercity travel | Sleeper‑class train (Delhi‑Mumbai) for 3 pax, round‑trip (1,200 × 6) | **7,200** | Assumed INR 1,200 per person one‑way (incl. reservation). |
| Accommodation | 2 nights in a budget hotel/guesthouse, triple occupancy (double room + extra bed) | **3,000** | Assumed INR 1,500 per nig

In [21]:
print("\n=== ToT Replanned Result ===")
print(tot_replanned_result["final_answer"])


=== ToT Replanned Result ===
## Candidate Plans
**Branch A – Lowest‑cost / Budget‑first**  
- Travel: Sleeper‑class train (Delhi ↔ Mumbai, 3 tickets each way) – ≈ INR 7,200  
- Accommodation: Budget guesthouse/dorm, 2 nights – ≈ INR 2,400  
- Meals: INR 200 per person per day × 3 days × 3 travellers – ≈ INR 1,800  
- Local transport: Mumbai trains & occasional auto – ≈ INR 300  
- Sightseeing: Basic tickets (Elephanta, museum) – ≈ INR 1,000  
- Emergency buffer (5 % of subtotal) – ≈ INR 1,000  
**Estimated total:** **INR 13,700**  

**Branch B – Balanced cost & experience**  
- Travel: AC‑3‑tier train (Delhi ↔ Mumbai, 3 tickets each way) – ≈ INR 12,000  
- Accommodation: Mid‑range hotel (double room + extra bed), 2 nights – ≈ INR 4,000  
- Meals: INR 300 per person per day × 3 days × 3 travellers – ≈ INR 2,700  
- Local transport: Mumbai trains, auto, ferry – ≈ INR 500  
- Sightseeing: Elephanta ferry, museum, temple donations – ≈ INR 1,500  
- Emergency buffer (5 % of subtotal) – ≈ I

In [22]:
print("\n=== ReAct Replanned Result ===")
print(react_replanned_result["final_answer"])


=== ReAct Replanned Result ===
## Tool Evidence
I conducted web searches for:
- Mumbai winter attractions and budget sightseeing options
- Delhi to Mumbai travel costs (bus, train, flight) 
- General India travel budget guidelines
- Mumbai-specific winter activities and free attractions

Key findings:
- Delhi-Mumbai bus: INR 1,500-8,000 per person (32-34 hours)
- Budget accommodation: INR 800-1,500 per night for shared rooms
- Mumbai has many free attractions: Marine Drive, Gateway of India, Colaba Causeway, Juhu Beach, Siddhivinayak Temple
- Local transport via Mumbai trains: INR 40-60 per journey
- Used budget calculator to create realistic cost breakdown

## Budget Breakdown
| Category | Cost (INR) | Details |
|----------|------------|---------|
| Return Travel | 4,500 | INR 1,500 per person × 3 travelers (budget overnight bus) |
| Accommodation | 2,400 | INR 800 per night × 3 nights (shared triple room in budget guesthouse) |
| Meals | 3,600 | INR 400 per person per day × 3 days ×

## Evaluation Framework

Each initial and replanned itinerary is evaluated using the same rubric:

- Budget compliance
- Traveller and duration constraints
- Practicality of itinerary
- Overall plan quality
- Dynamic replanning success

In [23]:
import pandas as pd


EVALUATOR_SYSTEM_PROMPT = """
You are a strict evaluator for AI-generated travel itineraries.

Evaluate only the plan provided. Do not invent missing facts.
Return valid JSON only.
"""


def evaluate_plan(plan_text, budget_limit):
    evaluation_prompt = f"""
Evaluate the following travel plan.

Budget limit: INR {budget_limit}

TRAVEL PLAN:
{plan_text}

Return a JSON object with exactly these fields:

{{
  "estimated_total_inr": number or null,
  "within_budget": true or false,
  "has_three_day_itinerary": true or false,
  "supports_three_travellers": true or false,
  "practicality_score_out_of_10": number,
  "overall_quality_score_out_of_10": number,
  "short_justification": "short explanation"
}}

Scoring guide:
- 9-10: complete, realistic, budget-aware, and practical
- 6-8: mostly good but has minor gaps or unclear assumptions
- 0-5: unrealistic, incomplete, over-budget, or missing key constraints
"""

    response = co.chat(
        model="command-a-plus-05-2026",
        messages=[
            {"role": "system", "content": EVALUATOR_SYSTEM_PROMPT},
            {"role": "user", "content": evaluation_prompt}
        ],
        response_format={"type": "json_object"},
        temperature=0
    )

    return json.loads(extract_response_text(response))

In [24]:
evaluation_results = {
    "CoT - Initial": evaluate_plan(
        cot_initial_result["final_answer"],
        budget_limit=40000
    ),
    "ToT - Initial": evaluate_plan(
        tot_initial_result["final_answer"],
        budget_limit=40000
    ),
    "ReAct - Initial": evaluate_plan(
        react_initial_result["final_answer"],
        budget_limit=40000
    ),
    "CoT - Replanned": evaluate_plan(
        cot_replanned_result["final_answer"],
        budget_limit=32000
    ),
    "ToT - Replanned": evaluate_plan(
        tot_replanned_result["final_answer"],
        budget_limit=32000
    ),
    "ReAct - Replanned": evaluate_plan(
        react_replanned_result["final_answer"],
        budget_limit=32000
    )
}

In [25]:
comparison_rows = [
    {
        "Strategy": "CoT",
        "Scenario": "Initial",
        "Reasoning / Planning Steps": cot_initial_result["planning_steps"],
        "Tool Calls": 0,
        "Execution Time (s)": cot_initial_result["execution_time_seconds"],
        **evaluation_results["CoT - Initial"]
    },
    {
        "Strategy": "ToT",
        "Scenario": "Initial",
        "Reasoning / Planning Steps": tot_initial_result["planning_steps"],
        "Tool Calls": 0,
        "Execution Time (s)": tot_initial_result["execution_time_seconds"],
        **evaluation_results["ToT - Initial"]
    },
    {
        "Strategy": "ReAct",
        "Scenario": "Initial",
        "Reasoning / Planning Steps": "Tool-driven",
        "Tool Calls": react_initial_result["tool_calls_count"],
        "Execution Time (s)": react_initial_result["execution_time_seconds"],
        **evaluation_results["ReAct - Initial"]
    },
    {
        "Strategy": "CoT",
        "Scenario": "Replanned",
        "Reasoning / Planning Steps": cot_replanned_result["planning_steps"],
        "Tool Calls": 0,
        "Execution Time (s)": cot_replanned_result["execution_time_seconds"],
        **evaluation_results["CoT - Replanned"]
    },
    {
        "Strategy": "ToT",
        "Scenario": "Replanned",
        "Reasoning / Planning Steps": tot_replanned_result["planning_steps"],
        "Tool Calls": 0,
        "Execution Time (s)": tot_replanned_result["execution_time_seconds"],
        **evaluation_results["ToT - Replanned"]
    },
    {
        "Strategy": "ReAct",
        "Scenario": "Replanned",
        "Reasoning / Planning Steps": "Tool-driven",
        "Tool Calls": react_replanned_result["tool_calls_count"],
        "Execution Time (s)": react_replanned_result["execution_time_seconds"],
        **evaluation_results["ReAct - Replanned"]
    }
]

comparison_df = pd.DataFrame(comparison_rows)

comparison_df

,Strategy,Scenario,Reasoning / Planning Steps,Tool Calls,Execution Time (s),estimated_total_inr,within_budget,has_three_day_itinerary,supports_three_travellers,practicality_score_out_of_10,overall_quality_score_out_of_10,short_justification
0,CoT,Initial,4,0,9.41,25840,True,True,True,9,9,"The plan is complete, realistic, and stays wel..."
1,ToT,Initial,6,0,32.74,28560,True,True,True,9,9,"Branch B meets the INR 40,000 budget, supports..."
2,ReAct,Initial,Tool-driven,4,14.43,17000,True,True,True,9,9,"Realistic budget breakdown covering travel, ac..."
3,CoT,Replanned,4,0,22.04,17600,True,True,True,8,8,"Meets all constraints, under budget, realistic..."
4,ToT,Replanned,6,0,15.46,22200,True,True,True,9,9,Branch B offers a realistic three‑day plan for...
5,ReAct,Replanned,Tool-driven,4,12.80,14000,True,True,True,8,7,"Meets budget and traveler count, three‑day iti..."


# Comparison Report: CoT vs ToT vs ReAct for Autonomous Trip Planning

## Objective
This assignment compares three reasoning strategies for planning a 3-day winter trip from Delhi to Mumbai for 3 travellers within a strict budget. The task also tests how well each strategy adapts when the budget drops from INR 40,000 to INR 32,000 mid-planning.

## Summary of Results

| Strategy | Strengths | Weaknesses | Performance under disruption |
|---|---|---|---|
| Chain-of-Thought (CoT) | Clear step-by-step reasoning, easy to follow, low overhead | Explores only one path; may ignore better alternatives | Replans logically but often with limited flexibility and fewer alternatives |
| Tree-of-Thought (ToT) | Considers multiple candidate plans and picks the strongest one | More reasoning and computation, higher complexity | Strongest at adaptation because it can compare alternatives under the new budget |
| ReAct | Uses tools and reacts to new information; realistic iterative planning | Can become noisy or less structured if tool calls are overused | Good for real-time replanning because it gathers fresh information and updates decisions |

## Analysis
The CoT strategy produced a coherent and structured itinerary because it followed a single linear plan: requirement analysis, budget allocation, itinerary creation, and final validation. This makes it suitable for straightforward tasks with clear constraints and no major uncertainty.

The ToT strategy performed better when the problem became more uncertain. Instead of committing to one path immediately, it generated multiple plan branches such as budget-first, balanced, and comfort-first options, then evaluated them against travel time, accommodation quality, food costs, local transport, and overall feasibility. This made it highly effective when constraints changed mid-task.

The ReAct strategy was strongest in dynamic, tool-driven planning. By using web search and budget-calculation tools, it could gather live information and revise plans based on current availability or practical constraints. This made it flexible and adaptive, especially in real-world scenarios where information can change during execution.

## Dynamic Replanning Observation
When the available budget was reduced from INR 40,000 to INR 32,000, all three strategies attempted a revised plan. However, the quality of the replanning differed:

- CoT adjusted the original structure but mostly reduced comfort and buffer spending.
- ToT re-evaluated alternative routes and budget allocations, producing a more robust revised itinerary.
- ReAct adapted by incorporating new information and tool-driven reasoning, which made the response more practical and context-aware.

## Recommendation
- Use CoT when the task is well-defined, deterministic, and needs a simple, explainable plan.
- Use ToT when the task involves trade-offs, multiple possible solutions, or significant uncertainty.
- Use ReAct when the problem requires live information retrieval, tool usage, and iterative adjustment during execution.

## Final Conclusion
For autonomous trip planning, ToT is the best choice when the task needs strong decision-making under changing constraints, while ReAct is best for real-world tasks that require active information gathering and adaptation. CoT remains the most transparent and simple strategy, but it is less flexible under disruption. In this assignment, ToT and ReAct were stronger for dynamic replanning, while CoT was effective for the initial baseline plan.

## Student Reflection
This project demonstrates that reasoning strategy matters as much as the task itself. A single-step approach may be enough for simple planning, but real-world AI agents need adaptive reasoning when budgets, constraints, and information change. The comparison clearly shows that robustness and replanning ability are essential in autonomous decision-making systems.